### Imports

In [12]:
import h5py
import numpy as np
from scipy.signal import welch
import os
import glob
import re
from pathlib import Path
import pandas as pd
import librosa
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.model_selection import StratifiedKFold, LeaveOneGroupOut
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

### Mappings

In [ ]:
MIC_CHANNELS = {
    "TrailK1": ["TrailK1", "Ch_1_labV12"],
    "TrailK2": ["TrailK2", "Ch_2_labV12"],
    "LeadK1":  ["LeadK1", "Ch_3_labV12"],
    "LeadK2":  ["LeadK2", "Ch_4_labV12"],
    "NAWSSound": ["NAWSSound"],
    "mic_iso": ["mic_iso", "MikrofonOst"],
    "mic_2m": ["mic_2m", "MikrofonWest"],
}

TRACKS = ["track150", "track211"]
CARS = ["01_ID.4", "02_Q8 e-tron", "03_Taycan", "04_E-Golf"]
TYRES = ["tyre1", "tyre3", "tyre6", "tyre10", "tyre12", "tyre13"]

### Feature Extraction Helper

In [3]:
def extract_audio_features_from_signal(signal, fs, n_mfcc=13, roll_percent=0.95):
    """
    Extract single-scalar audio features from a 1D signal.

    Time-domain:
        - rms, mean, std, max, crest, zcr

    Frequency-domain (Welch PSD):
        - spec_centroid
        - spec_rolloff (e.g. 95% of spectral energy)
        - spec_flatness (geometric mean / arithmetic mean of PSD)
        - spec_bandwidth (spectral spread around centroid)
        - band_i (absolute band powers over fixed frequency ranges)

    MFCC (if librosa is available):
        - mfcc_1 ... mfcc_n_mfcc (mean over time for each coefficient)
    """
    if signal is None or fs is None:
        return {}

    # Make sure signal is 1D float
    x = np.asarray(signal, dtype=float).squeeze()
    if x.ndim != 1 or x.size < 10:
        # degenerate / empty channel
        return {}

    feats = {}

    # --- Time-domain ---
    feats["rms"] = float(np.sqrt(np.mean(x**2)))
    feats["mean"] = float(np.mean(x))
    feats["std"] = float(np.std(x))
    feats["max"] = float(np.max(np.abs(x)))
    feats["crest"] = float(feats["max"] / (feats["rms"] + 1e-9))

    # zero-crossing rate
    feats["zcr"] = float(((x[:-1] * x[1:]) < 0).mean())

    # --- Frequency-domain using Welch PSD ---
    f, psd = welch(x, fs, nperseg=min(4096, x.size))

    # ensure 1D PSD
    psd = np.asarray(psd).squeeze()
    f = np.asarray(f).squeeze()

    if f.ndim != 1 or psd.ndim != 1 or f.size != psd.size:
        # something weird, bail out on frequency features
        return feats

    total_power = np.sum(psd) + 1e-9

    # Spectral centroid (power-weighted mean frequency)
    spec_centroid = np.sum(f * psd) / total_power
    feats["spec_centroid"] = float(spec_centroid)

    # --- NEW: Spectral rolloff ---
    # Frequency below which `roll_percent` of total spectral energy lies.
    # Common choice: roll_percent = 0.95
    cumulative_power = np.cumsum(psd)
    threshold = roll_percent * cumulative_power[-1]
    idx = np.searchsorted(cumulative_power, threshold)
    idx = min(idx, len(f) - 1)
    feats["spec_rolloff"] = float(f[idx])

    # --- NEW: Spectral flatness ---
    # Ratio of geometric mean to arithmetic mean of PSD.
    # ~1 → noise-like, <<1 → tone-like.
    psd_safe = psd + 1e-12  # avoid log(0) / division by zero
    geo_mean = float(np.exp(np.mean(np.log(psd_safe))))
    arith_mean = float(np.mean(psd_safe))
    feats["spec_flatness"] = float(geo_mean / (arith_mean + 1e-12))

    # --- NEW: Spectral bandwidth (spread around centroid) ---
    # Here: standard deviation of frequency around centroid, weighted by PSD.
    feats["spec_bandwidth"] = float(
        np.sqrt(np.sum(((f - spec_centroid) ** 2) * psd) / total_power)
    )

    # --- Band powers (already present, kept as-is) ---
    bands = [(0, 200), (200, 500), (500, 1000), (1000, 2000), (2000, 5000)]
    for i, (lo, hi) in enumerate(bands):
        mask = (f >= lo) & (f < hi)
        if not np.any(mask):
            feats[f"band_{i}"] = 0.0
        else:
            feats[f"band_{i}"] = float(np.sum(psd[mask]))

    # --- NEW: MFCC features (single scalars per coefficient) ---
    # We use librosa if available. Each coefficient is averaged over time.
    # librosa expects float32 and mono; x is already 1D float64
    mfcc = librosa.feature.mfcc(
        y=x.astype(np.float32),
        sr=fs,
        n_mfcc=n_mfcc
    )  # shape: (n_mfcc, n_frames)

    mfcc_means = mfcc.mean(axis=1)  # mean over time for each coeff
    for i, coeff in enumerate(mfcc_means, start=1):
        feats[f"mfcc_{i}"] = float(coeff)

    return feats

def extract_all_audio_features_from_h5(path):
    with h5py.File(path, "r") as f:
        all_feats = {}

        for mic_name, synonyms in MIC_CHANNELS.items():

            for synonym in synonyms:
                if synonym in f:
                    sig = f[synonym][:]
                    fs = f[synonym].attrs.get("sample_rate", None)

                    # sample_rate is sometimes an array, make it scalar float
                    if fs is not None:
                        fs = float(np.array(fs).ravel()[0])
                    
                    feats = extract_audio_features_from_signal(sig, fs)

                    for k, v in feats.items():
                        all_feats[f"{mic_name}_{k}"] = v

                    break
            
            else:
                print(f"Warning: No channel found for {mic_name} in {path}")

        return all_feats
    

# Filename metadata parsing (track type, car id, tyre id)
def parse_metadata_from_filename(path):
    fname = os.path.basename(path)
    stem = os.path.splitext(fname)[0]  # remove ".h5"

    # Example: track211_E-Golf_tyre13_meas1_2p5_1_2025-09-26_14-48-57
    m = re.match(r"track(\d+)_(.+?)_tyre(\d+)_", stem)
    if not m:
        # fallback: return minimal info if pattern doesn't match
        print(f"Could not parse metadata from filename because pattern did not match: {fname}")
        return {
            "road_type": None,
            "car_id": None,
            "tyre_id": None,
            "filename": fname,
        }

    road_type = m.group(1)        # 150 or 211
    car_name = m.group(2)         # e.g. "ID.4"
    tyre_id = m.group(3)          # e.g. "13"

    return {
        "road_type": road_type,
        "car_id": car_name,
        "tyre_id": tyre_id,
        "filename": fname,
    }


### Build dataset

In [26]:
# Get the h5 file paths
DATA_ROOT = r"C:\Users\Lars\Büro\KIT\Master\WS_25_26\AIFB_Seminar\projects\workspace\data"

h5_paths = glob.glob(os.path.join(DATA_ROOT, "**", "*.h5"), recursive=True)
print(f"Found {len(h5_paths)} files")

rows = []

for path in h5_paths:
    try:
        feat_dict = extract_all_audio_features_from_h5(path)
        if not feat_dict:
            print(f"No usable mic data in {path}, skipping.")
            continue  # no usable mic data

        meta = parse_metadata_from_filename(path)

        row = {
            **feat_dict,
            **meta,
            "filepath": path,
        }
        rows.append(row)

    except Exception as e:
        print(f"Error processing {path}: {e}")

df = pd.DataFrame(rows)

df.info(verbose=True, show_counts=True)

Found 204 files
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 204 entries, 0 to 203
Data columns (total 173 columns):
 #    Column                    Non-Null Count  Dtype  
---   ------                    --------------  -----  
 0    TrailK1_rms               204 non-null    float64
 1    TrailK1_mean              204 non-null    float64
 2    TrailK1_std               204 non-null    float64
 3    TrailK1_max               204 non-null    float64
 4    TrailK1_crest             204 non-null    float64
 5    TrailK1_zcr               204 non-null    float64
 6    TrailK1_spec_centroid     204 non-null    float64
 7    TrailK1_spec_rolloff      204 non-null    float64
 8    TrailK1_spec_flatness     204 non-null    float64
 9    TrailK1_spec_bandwidth    204 non-null    float64
 10   TrailK1_band_0            204 non-null    float64
 11   TrailK1_band_1            204 non-null    float64
 12   TrailK1_band_2            204 non-null    float64
 13   TrailK1_band_3            204 no

### Fix missing values

1. mic_iso channels 151/204 non-null

    track211 ID.4 tyre3 is missing mic iso, remove it completely for now, contains however "MikrofonOst" & "MikrofonWest" with similar sample rates (btw regular files also contain a "mic_2m" additionally to "mic_iso") which could be used

2. NAWSSound channels 199/204 non-null

    5 track211 Q8 e-tron tyre12 files are missing the NAWS channel, fill it with means of the respective group

In [6]:
df_clean = df.copy()

# Drop the mic_iso channels
df_clean = df_clean.drop(columns=[col for col in df_clean.columns if col.startswith("mic_iso")])

# Fill missing NAWSSound features with group-wise mean imputation
naws_cols = [col for col in df_clean.columns if col.startswith("NAWSSound")]
group_cols = ["road_type", "car_id", "tyre_id"]

for col in naws_cols:
    df_clean[col] = df_clean.groupby(group_cols)[col].transform(
        lambda g: g.fillna(g.mean())
    )

# If some groups have only NaN (rare), fill remaining with global mean
if df_clean[naws_cols].isna().any().any():
    print("Some NAWSSound groups have only NaN, filling with global mean.")
    df_clean[naws_cols] = df_clean[naws_cols].fillna(df_clean[naws_cols].mean())

df_clean.info(verbose=True, show_counts=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 204 entries, 0 to 203
Data columns (total 145 columns):
 #    Column                    Non-Null Count  Dtype  
---   ------                    --------------  -----  
 0    TrailK1_rms               204 non-null    float64
 1    TrailK1_mean              204 non-null    float64
 2    TrailK1_std               204 non-null    float64
 3    TrailK1_max               204 non-null    float64
 4    TrailK1_crest             204 non-null    float64
 5    TrailK1_zcr               204 non-null    float64
 6    TrailK1_spec_centroid     204 non-null    float64
 7    TrailK1_spec_rolloff      204 non-null    float64
 8    TrailK1_spec_flatness     204 non-null    float64
 9    TrailK1_spec_bandwidth    204 non-null    float64
 10   TrailK1_band_0            204 non-null    float64
 11   TrailK1_band_1            204 non-null    float64
 12   TrailK1_band_2            204 non-null    float64
 13   TrailK1_band_3            204 non-null    float6

### Create LOGO Split

In [7]:
# Explicitly exclude non-feature columns (safer than dtype selection alone)
non_feature_cols = ["road_type", "car_id", "tyre_id", "filename", "filepath"]
feature_cols = [c for c in df_clean.columns
                if c not in non_feature_cols and df_clean[c].dtype == "float64"]

print("Number of features:", len(feature_cols))

X = df_clean[feature_cols].to_numpy()
y = df_clean["road_type"].to_numpy()
groups = df_clean["car_id"].to_numpy()

# Encode string labels to integers for XGBoost
le = LabelEncoder()
y_enc = le.fit_transform(y)
print("Classes:", le.classes_)

# LOGO splitter
logo = LeaveOneGroupOut()

Number of features: 140
Classes: ['150' '211']


### Balancing Helper

In [14]:
def car_balanced_weights(car_ids):
    counts = Counter(car_ids)
    w = np.array([1.0 / counts[c] for c in car_ids], dtype=float)
    # normalize for nicer scales (optional)
    return w * (len(w) / w.sum())

### Training Loop

In [36]:
# Config

CLASS_BALANCING = False
CAR_BALANCING = True
THRESHOLD_TUNING = False     # Only for Random Forest

In [37]:
rf_accs, rf_f1s = [], []
xgb_accs, xgb_f1s = [], []
fold_results = []

# Optional: to accumulate global confusion matrices across folds
# (nice for seminar slides)
labels = le.classes_
rf_cm_total = np.zeros((len(labels), len(labels)), dtype=int)
xgb_cm_total = np.zeros((len(labels), len(labels)), dtype=int)

for fold, (train_idx, test_idx) in enumerate(logo.split(X, y, groups=groups), start=1):
    held_out_car = groups[test_idx][0]
    print(f"\n===== LOGO Fold {fold} | Held-out car: {held_out_car} =====")
    print(f"Train n={len(train_idx)}, Test n={len(test_idx)}")

    # Split data (enc for XGBoost)
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]
    y_train_enc, y_test_enc = y_enc[train_idx], y_enc[test_idx]

    # Compute class weights
    class_weights = compute_sample_weight("balanced", y_train)

    # Compute car-based weights to balance training samples by car
    car_train = df_clean.iloc[train_idx]["car_id"].to_numpy()
    car_weights = car_balanced_weights(car_train)

    # ------------------ Random Forest ------------------
    rf = RandomForestClassifier(
        n_estimators=300,
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1,
        n_jobs=-1,
        random_state=42
    )

    # Fit with appropriate sample weights
    if CLASS_BALANCING and CAR_BALANCING:
        combined_weights = class_weights * car_weights
        combined_weights = combined_weights * (len(combined_weights) / combined_weights.sum())
        rf.fit(X_train, y_train, sample_weight=combined_weights)
    elif CLASS_BALANCING:
        rf.fit(X_train, y_train, sample_weight=class_weights)
    elif CAR_BALANCING:
        rf.fit(X_train, y_train, sample_weight=car_weights)
    else:
        rf.fit(X_train, y_train)

    # Threshold tuning
    if THRESHOLD_TUNING:
        proba0 = rf.predict_proba(X_test)[:, 0]
        t0 = 0.3  # threshold for class 0
        y_pred_rf_enc = (proba0 < t0).astype(int)
        y_pred_rf = le.inverse_transform(y_pred_rf_enc)
    else:
        y_pred_rf = rf.predict(X_test)

    rf_acc = accuracy_score(y_test, y_pred_rf)
    rf_f1 = f1_score(y_test, y_pred_rf, average="weighted")

    rf_accs.append(rf_acc)
    rf_f1s.append(rf_f1)

    cm_rf = confusion_matrix(y_test, y_pred_rf, labels=labels)
    rf_cm_total += cm_rf

    print(f"RF  - acc: {rf_acc:.3f}, f1: {rf_f1:.3f}")
    print(cm_rf)

    # ------------------ XGBoost ------------------
    # For 2-class problems we can use binary:logistic.
    # If we ever add more road types, switch to multi:softprob + num_class.
    xgb = XGBClassifier(
        n_estimators=400,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="binary:logistic",
        n_jobs=-1,
        eval_metric="logloss",
        random_state=42
    )

    if CLASS_BALANCING and CAR_BALANCING:
        xgb.fit(X_train, y_train_enc, sample_weight=combined_weights)
    elif CLASS_BALANCING:
        xgb.fit(X_train, y_train_enc, sample_weight=class_weights)
    elif CAR_BALANCING:
        xgb.fit(X_train, y_train_enc, sample_weight=car_weights)
    else:
        xgb.fit(X_train, y_train_enc)

    y_pred_xgb_enc = xgb.predict(X_test)
    y_pred_xgb = le.inverse_transform(y_pred_xgb_enc.astype(int))

    xgb_acc = accuracy_score(y_test, y_pred_xgb)
    xgb_f1 = f1_score(y_test, y_pred_xgb, average="weighted")

    xgb_accs.append(xgb_acc)
    xgb_f1s.append(xgb_f1)

    cm_xgb = confusion_matrix(y_test, y_pred_xgb, labels=labels)
    xgb_cm_total += cm_xgb

    print(f"XGB - acc: {xgb_acc:.3f}, f1: {xgb_f1:.3f}")
    print(cm_xgb)

    fold_results.append({
        "fold": fold,
        "held_out_car": held_out_car,
        "n_test": len(test_idx),
        "rf_acc": rf_acc,
        "rf_f1": rf_f1,
        "xgb_acc": xgb_acc,
        "xgb_f1": xgb_f1
    })


# --------------- Summary ---------------
print("\n===== LOGO Summary (per held-out car) =====")
for r in fold_results:
    print(f"{r['held_out_car']}: RF acc={r['rf_acc']:.3f}, XGB acc={r['xgb_acc']:.3f}")

print("\n===== Overall (mean ± std) =====")
print(f"RF  acc: {np.mean(rf_accs):.3f} ± {np.std(rf_accs):.3f} | "
      f"f1: {np.mean(rf_f1s):.3f} ± {np.std(rf_f1s):.3f}")
print(f"XGB acc: {np.mean(xgb_accs):.3f} ± {np.std(xgb_accs):.3f} | "
      f"f1: {np.mean(xgb_f1s):.3f} ± {np.std(xgb_f1s):.3f}")

print("\nRF total confusion matrix (rows=true, cols=pred):")
print(rf_cm_total)
print("\nXGB total confusion matrix (rows=true, cols=pred):")
print(xgb_cm_total)
print("\nLabel order:", labels)


===== LOGO Fold 1 | Held-out car: E-Golf =====
Train n=179, Test n=25
RF  - acc: 0.720, f1: 0.717
[[ 6  4]
 [ 3 12]]
XGB - acc: 0.640, f1: 0.636
[[ 5  5]
 [ 4 11]]

===== LOGO Fold 2 | Held-out car: ID.4 =====
Train n=115, Test n=89
RF  - acc: 0.910, f1: 0.876
[[ 1  8]
 [ 0 80]]
XGB - acc: 0.876, f1: 0.865
[[ 2  7]
 [ 4 76]]

===== LOGO Fold 3 | Held-out car: Q8 e-tron =====
Train n=141, Test n=63
RF  - acc: 0.714, f1: 0.595
[[ 0 18]
 [ 0 45]]
XGB - acc: 0.683, f1: 0.669
[[ 6 12]
 [ 8 37]]

===== LOGO Fold 4 | Held-out car: Taycan =====
Train n=177, Test n=27
RF  - acc: 0.667, f1: 0.533
[[ 0  9]
 [ 0 18]]
XGB - acc: 0.667, f1: 0.533
[[ 0  9]
 [ 0 18]]

===== LOGO Summary (per held-out car) =====
E-Golf: RF acc=0.720, XGB acc=0.640
ID.4: RF acc=0.910, XGB acc=0.876
Q8 e-tron: RF acc=0.714, XGB acc=0.683
Taycan: RF acc=0.667, XGB acc=0.667

===== Overall (mean ± std) =====
RF  acc: 0.753 ± 0.093 | f1: 0.681 ± 0.131
XGB acc: 0.716 ± 0.094 | f1: 0.676 ± 0.120

RF total confusion matrix (r

### Feature Importance

In [38]:
rf_final = RandomForestClassifier(
    n_estimators=500,
    max_depth=None,
    n_jobs=-1,
    random_state=42
)

# Compute class weights
class_weights_final = compute_sample_weight("balanced", y)

# Compute car-based weights to balance training samples by car
car_final = df_clean["car_id"].to_numpy()
car_weights_final = car_balanced_weights(car_final)
if CLASS_BALANCING and CAR_BALANCING:
    combined_weights_final = class_weights_final * car_weights_final
    combined_weights_final = combined_weights_final * (len(combined_weights_final) / combined_weights_final.sum())
    rf_final.fit(X, y, sample_weight=combined_weights_final)
elif CLASS_BALANCING:
    rf_final.fit(X, y, sample_weight=class_weights_final)
elif CAR_BALANCING:
    rf_final.fit(X, y, sample_weight=car_weights_final)
else:
    rf_final.fit(X, y)

importances_rf = pd.Series(rf_final.feature_importances_, index=feature_cols)
print("\nTop 20 features Random Forest:")
print(importances_rf.sort_values(ascending=False).head(20))


Top 20 features Random Forest:
NAWSSound_mfcc_10          0.037143
NAWSSound_band_0           0.024299
LeadK1_mfcc_12             0.022223
NAWSSound_spec_centroid    0.021599
NAWSSound_mfcc_13          0.020387
TrailK2_mfcc_2             0.019499
TrailK1_mfcc_13            0.018250
NAWSSound_mfcc_2           0.018222
LeadK1_mfcc_13             0.017088
LeadK2_mfcc_12             0.015998
NAWSSound_crest            0.015975
NAWSSound_mfcc_9           0.015672
LeadK2_mfcc_13             0.015437
TrailK1_mfcc_2             0.014723
TrailK1_mfcc_12            0.013722
LeadK2_mfcc_9              0.013600
NAWSSound_zcr              0.013439
LeadK1_mfcc_3              0.013212
TrailK2_mfcc_4             0.013056
TrailK1_mfcc_4             0.013041
dtype: float64


In [39]:
xgb_final = XGBClassifier(
    n_estimators=400,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    n_jobs=-1,
    eval_metric="logloss",
    random_state=42
)

if CLASS_BALANCING and CAR_BALANCING:
    xgb_final.fit(X, y_enc, sample_weight=combined_weights_final)
elif CLASS_BALANCING:
    xgb_final.fit(X, y_enc, sample_weight=class_weights_final)
elif CAR_BALANCING:
    xgb_final.fit(X, y_enc, sample_weight=car_weights_final)
else:
    xgb_final.fit(X, y_enc)

xgb_feature_scores = xgb_final.get_booster().get_score(importance_type='gain')
    
# Convert to pandas Series
importances_xgb = (
    pd.Series(xgb_feature_scores)
    .rename(index=lambda x: feature_cols[int(x[1:])])  # maps f0 → feature name
    .sort_values(ascending=False)
)

xgb_norm = importances_xgb / importances_xgb.sum()
print("\nTop 20 features XGBoost:")
print(xgb_norm.head(20))


Top 20 features XGBoost:
TrailK2_max               0.037236
TrailK1_mfcc_13           0.036557
LeadK2_max                0.032552
TrailK2_rms               0.031049
NAWSSound_band_0          0.029021
LeadK1_mfcc_4             0.027028
LeadK1_mfcc_3             0.022739
NAWSSound_mfcc_4          0.022350
NAWSSound_mfcc_10         0.020568
TrailK1_max               0.019888
LeadK2_mfcc_6             0.019579
NAWSSound_mfcc_7          0.018879
TrailK1_band_2            0.017489
LeadK1_mfcc_7             0.017414
LeadK2_band_3             0.017412
LeadK1_band_1             0.016998
TrailK1_spec_bandwidth    0.016332
LeadK1_mfcc_2             0.016306
TrailK2_mfcc_2            0.016156
LeadK1_mfcc_10            0.016130
dtype: float64
